In [ ]:
#uv add langchain langchain-community langchain-teddynote langchain-openai

In [ ]:
#uv add langchain-classic langsmith langchain-chroma pypdf pydantic

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("Summary")

loader = TextLoader(
    "data/news.txt",
    encoding="utf-8")

docs = loader.load()
print(f"총 글자수 : {len(docs[0].page_content)}")
print("\n========= 앞부분 미리보기 =========\n")
print(docs[0].page_content[:500])

LangSmith 추적을 시작합니다.
[프로젝트명]
Summary
총 글자수 : 1015

========= 앞부분 미리보기 =========

정점식 "일도 못하는 무능한 대통령…국민 한숨 이어져"

인사·부동산·안보·사법 등 전방위 비판
"DMZ 지뢰 사고 은폐…국회 청문회·국정조사해야"


매일일보 = 조정훈 인턴기자  |  정점식 국민의힘 원내대표는 28일 "이번 추석 연휴 막판 민심은 한마디로 분노와 한숨이었다. 도덕성은 기대도 안 했고 일은 좀 하나 했는데 알고 보니 일도 못 하는 무능한 대통령"이라고 말했다.

정 원내대표는 이날 최고위원회의에서 추석에 나타난 이재명 정부에 대한 민심으로 △계속되는 인사 참사의 원인과 책임에 대해 성남·경기 라인, 비선 실세의 인사 농단 진상 규명 요구 △부동산·농지·주식 등 국민의 재산을 약탈하는 정권에 대한 분노 등을 거론하며 이같이 지적했다.

그는 △일주일도 남지 않은 검찰 해체, 중대범죄수사청 출범, 보완수사권 폐지를 앞두고 치안과 범죄 수사 당국의 역량에 대한 불안 고조 △사법부 장악 및 삼권분립 파괴 걱정 △비무장지대(DMZ) 지뢰 폭발에 대한 정부의 납득하기 힘든 


In [8]:
from langsmith import Client

client = Client()

prompt = client.pull_prompt(
    "teddynote/summary-stuff-documents-korean",
    dangerously_pull_public_prompt=True)
prompt.pretty_print()

Please summarize the sentence according to the following REQUEST.
REQUEST:
1. Summarize the main points in bullet points in KOREAN.
2. Each summarized sentence must start with an emoji that fits the meaning of the each sentence.
3. Use various emojis to make the summary more interesting.
4. Translate the summary into KOREAN if it is written in ENGLISH.
5. DO NOT translate any technical terms.
6. DO NOT include any unnecessary information.

CONTEXT:
{context}

SUMMARY:"



In [10]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_teddynote.callbacks import StreamingCallback

llm = ChatOpenAI(
    model_name="gpt-4o-mini",
    streaming=True,
    temperature=0.1,
    callbacks=[StreamingCallback()],
)

stuff_chain = create_stuff_documents_chain(llm, prompt)
answer = stuff_chain.invoke({"context": docs})

- 😡 정점식 원내대표는 대통령의 무능함에 대한 국민의 분노를 언급했다.
- 🔍 인사 참사와 비선 실세의 인사 농단에 대한 진상 규명을 요구했다.
- 💰 부동산과 국민 재산을 약탈하는 정권에 대한 분노를 표현했다.
- ⚖️ 사법부 장악과 삼권분립 파괴에 대한 우려를 제기했다.
- 💣 DMZ 지뢰 폭발 사고에 대한 정부의 대응을 비판하며 국정조사를 촉구했다.
- 🕵️‍♂️ 군의 조사 태도가 은폐와 암장에 가까웠다고 주장했다.
- 🔗 북한의 소행으로 거의 확실한 상황에서 국방부 장관의 태도를 비판했다.
- 🤔 이재명 대통령의 대북 제재 해제 주장에 혼란을 느낀다고 지적했다.

In [23]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader("data/Truth.pdf")
docs = loader.load()
docs = docs[0:9]
print(f"총 페이지수: {len(docs)}")

총 페이지수: 9


In [ ]:
from langsmith import Client
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

client = Client()

llm = ChatOpenAI(temperature=0.1, model_name = "gpt-4o-mini")

map_prompt = client.pull_prompt(
    "teddynote/map-prompt",
    dangerously_pull_public_prompt=True
)

map_prompt.pretty_print()

In [28]:
map_chain = map_prompt | llm | StrOutputParser()
doc_summaries = map_chain.batch(docs)
print(doc_summaries[5])

- 이재명 대통령은 미국과 멕시코 순방을 마친 후 알래스카주 앵커리지에서 기자들과 즉석 차담회를 가졌다.
- 이번 순방에서는 유엔총회 일정과 한미 정상회담, 멕시코 정상회담 등의 외교 일정이 포함되었다.
- 귀국 과정에서의 기자단과의 소통 장면을 중심으로 순방의 마무리 상황이 전달되었다.


In [ ]:
reduce_prompt = client.pull_prompt(
    "teddynote/reduce-prompt",
    dangerously_pull_public_prompt=True
)

reduce_prompt.pretty_print()

In [33]:
reduce_chain = reduce_prompt | llm | StrOutputParser()

from langchain_teddynote.messages import stream_response

answer = reduce_chain.stream(
    {"doc_summaries": "\n".join(doc_summaries), "language": "Korean"}
)

stream_response(answer)

- 매일일보 조정훈 인턴기자의 기사 7건을 바탕으로 핵심 내용을 재구성한 실습 자료이다.
- 이 자료는 RAG, VectorStore, Retriever와 같은 기술을 활용한 문서 분할 실습용으로 제공된다.
- 수집 기준일은 2026년 9월 29일이며, 주요 분야는 정치, 외교, 국정감사, 국방, 문화 논쟁이다.
- 야당이 국정감사를 앞두고 공세를 강화하고 있으며, 장외 집회에 대한 신중한 접근이 필요하다는 의견이 제기되고 있다.
- 2026년 국정감사에서 운영위원회, 재정경제위원회, 행정안전위원회가 주요 쟁점으로 부각되며, 인사, 부동산, 경찰 개혁이 핵심 논의 사항으로 예상된다.
- 미중 정상회담에서 양국 간의 웃음이 오갔지만, AI, 대만, 이란 문제에 대해서는 여전히 입장 차이가 존재한다.
- 국민의힘이 국정감사를 앞두고 정부에 대한 공세를 강화하고 있다.
- 당 내부에서는 장외투쟁의 필요성에 대한 공감이 있지만, 실제 집회 추진에는 신중한 태도가 보인다.
- 국정감사 대응과 장외투쟁 전략의 조합이 주요 쟁점으로 부각되고 있다.
- 2026년도 국정감사는 10월 6일부터 27일까지 3주 동안 진행될 예정이다.
- 운영위원회에서는 청와대 인사검증 논란이 주요 이슈로 제시된다.
- 재정경제 관련 상임위에서는 부동산 및 세제 현안이 다뤄질 예정이다.
- 행정안전위원회에서는 경찰개혁 문제가 핵심 쟁점으로 부각된다.
- 여러 상임위에서 다뤄질 현안을 통해 국정감사 전체 지형을 미리 파악할 수 있도록 구성되었다.
- 도널드 트럼프 미국 대통령과 시진핑 중국 국가주석의 정상회담에서 양국은 무역과 인공지능 분야의 협력을 강조했다.
- 그러나 AI, 대만, 이란 등 주요 현안에서는 여전히 입장차가 존재한다.
- 이재명 대통령은 미국과 멕시코 순방을 마친 후 알래스카주 앵커리지에서 기자들과 즉석 차담회를 가졌다.
- 정점식 국민의힘 원내대표는 추석 연휴 동안의 민심을 '분노와 한숨'으로 표현하며 이재명 대통령과 정부를 비판했다.
- 나경원 국민의힘 의원은 DMZ 지뢰 폭

In [36]:
from langchain_core.runnables import chain

@chain
def map_reduce_chain(docs):
    map_llm = ChatOpenAI(
        temperature=0.1, model_name = "gpt-4o-mini",
    )

    map_prompt = client.pull_prompt(
        "teddynote/map-prompt",
        dangerously_pull_public_prompt=True
    )
    
    map_chain = map_prompt | map_llm | StrOutputParser()
    
    doc_summaries = map_chain.batch(docs)
    
    reduce_prompt = client.pull_prompt(
        "teddynote/reduce-prompt",
        dangerously_pull_public_prompt=True
    )
    
    reduce_llm=ChatOpenAI(
        model_name="gpt-4o",
        temperature=0.1,
        callbacks=[StreamingCallback()],
        streaming=True,
    )
    
    reduce_chain = reduce_prompt | reduce_llm | StrOutputParser()
    
    return reduce_chain.invoke(
        {"doc_summaries": "\n".join(doc_summaries), "language": "Korean"}
    )

answer = map_reduce_chain.invoke(docs)

- 조정훈 인턴기자의 7건의 기사를 바탕으로 정치, 외교, 국정감사, 국방, 문화 논쟁 등의 주요 분야를 다루었다.
- 국정감사를 앞두고 야당은 공세 수위를 높이고 있으며, 장외 집회에 대한 신중론이 제기되고 있다.
- 2026년 국정감사에서는 운영위, 재경위, 행안위가 주요 쟁점으로 부각되며, 인사, 부동산, 경찰개혁이 주요 논의 사항이다.
- 미중 정상회담에서는 무역과 AI 협력을 강조했으나, AI, 대만, 이란 문제에 대한 입장 차이가 여전히 존재한다.
- 정점식 의원은 현 정부를 비판하며 국민의 불만을 강조했고, 나경원 의원은 DMZ 지뢰 폭발사고 조사에 야당 추천 전문가 포함을 주장했다.
- 국민의힘은 국정감사와 장외투쟁 전략을 조합하여 정부에 대한 공세를 강화하고 있다.
- 영화 '암살자(들)'이 역사적 사실 왜곡 논란을 일으키며 정치적 이슈로 확대되고 있다.
- 이재명 대통령은 미국과 멕시코 순방을 마치고 기자들과 소통하며 외교 일정을 마무리했다.